# ParallelBench: Experiment Results Visualization

Speed-quality tradeoff charts: **TPS (Tokens per Step)** vs **Accuracy (%)**  
Each chart shows one model with all unmasking methods as separate lines.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

## Filter Configuration

Set to `None` to include all. Provide a list to filter.

In [ ]:
# Filter configuration -- set to None to include all
FILTER_MODELS: list[str] | None = None  # e.g., ["GSAI-ML__LLaDA-1.5"]
FILTER_METHODS: list[str] | None = None  # e.g., ["confidence_topk", "random"]
FILTER_TASKS: list[str] | None = None  # e.g., ["parallelbench_all"]
OUTPUT_DIR = "../figures"  # PDF output directory

RESULTS_DIR = Path("../results")

## Data Loading

Auto-scan `results/` directory and build a DataFrame.  
TPS is always read from `tokens_per_step,none` metric (no dual-path branching).

In [ ]:
# Category-level groups to exclude (only keep individual tasks + avg)
CATEGORY_GROUPS = {
    "parallelbench_puzzles",
    "parallelbench_text_writing",
    "parallelbench_waiting_line",
}


def load_results(results_dir: Path) -> pd.DataFrame:
    """Scan results/ and build a DataFrame with columns: model, method, param, tps, task, score."""
    rows = []

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        model = model_dir.name

        for method_dir in sorted(model_dir.iterdir()):
            if not method_dir.is_dir():
                continue
            method = method_dir.name

            for param_dir in sorted(method_dir.iterdir()):
                if not param_dir.is_dir():
                    continue
                param = param_dir.name

                # Select latest timestamp directory (lexicographic sort)
                timestamp_dirs = sorted([d for d in param_dir.iterdir() if d.is_dir()])
                if not timestamp_dirs:
                    continue
                latest_dir = timestamp_dirs[-1]

                result_file = latest_dir / "results_parallelbench.json"
                if not result_file.exists():
                    continue

                with open(result_file) as f:
                    data = json.load(f)

                results = data.get("results", {})

                for task_name, task_metrics in results.items():
                    # Skip category-level groups
                    if task_name in CATEGORY_GROUPS:
                        continue

                    score = task_metrics.get("score,none")
                    tps = task_metrics.get("tokens_per_step,none")

                    if score is None or tps is None:
                        continue

                    rows.append(
                        {
                            "model": model,
                            "method": method,
                            "param": param,
                            "tps": float(tps),
                            "task": task_name,
                            "score": float(score),
                        }
                    )

    return pd.DataFrame(rows)


df = load_results(RESULTS_DIR)
print(
    f"Loaded {len(df)} rows from {df['model'].nunique()} models, {df['method'].nunique()} methods, {df['task'].nunique()} tasks"
)
df.head()

In [ ]:
# Apply filters
filtered_df = df.copy()
if FILTER_MODELS is not None:
    filtered_df = filtered_df[filtered_df["model"].isin(FILTER_MODELS)]
if FILTER_METHODS is not None:
    filtered_df = filtered_df[filtered_df["method"].isin(FILTER_METHODS)]
if FILTER_TASKS is not None:
    filtered_df = filtered_df[filtered_df["task"].isin(FILTER_TASKS)]

print(f"After filtering: {len(filtered_df)} rows")
print(f"Models: {sorted(filtered_df['model'].unique())}")
print(f"Methods: {sorted(filtered_df['method'].unique())}")
print(f"Tasks: {sorted(filtered_df['task'].unique())}")

## Style Configuration

In [ ]:
# Method display names
METHOD_DISPLAY_NAMES = {
    "confidence_topk": "Confidence Top-K",
    "confidence_threshold": "Confidence Threshold",
    "confidence_factor": "Confidence Factor",
    "entropy_topk": "Entropy Top-K",
    "topk_margin": "Top-K Margin",
    "random": "Random",
    "left_to_right": "Left-to-Right",
    "origin": "Origin",
    "klass": "KLASS",
}

# Method colors (inspired by parallelbench.github.io palette)
METHOD_COLORS = {
    "confidence_topk": "#2563eb",
    "confidence_threshold": "#7c3aed",
    "confidence_factor": "#db2777",
    "entropy_topk": "#059669",
    "topk_margin": "#d97706",
    "random": "#6b7280",
    "left_to_right": "#dc2626",
    "origin": "#0891b2",
    "klass": "#4f46e5",
}

# Method markers
METHOD_MARKERS = {
    "confidence_topk": "o",
    "confidence_threshold": "s",
    "confidence_factor": "D",
    "entropy_topk": "^",
    "topk_margin": "v",
    "random": "X",
    "left_to_right": "P",
    "origin": "*",
    "klass": "h",
}

# Fallback for unknown methods
_FALLBACK_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
_FALLBACK_MARKERS = ["o", "s", "D", "^", "v", "X", "P", "*", "h", "<", ">"]


def get_method_style(method: str, idx: int = 0) -> dict:
    """Return color, marker, and display name for a method."""
    return {
        "color": METHOD_COLORS.get(
            method, _FALLBACK_COLORS[idx % len(_FALLBACK_COLORS)]
        ),
        "marker": METHOD_MARKERS.get(
            method, _FALLBACK_MARKERS[idx % len(_FALLBACK_MARKERS)]
        ),
        "label": METHOD_DISPLAY_NAMES.get(method, method),
    }


# Model display names
MODEL_DISPLAY_NAMES = {
    "GSAI-ML__LLaDA-1.5": "LLaDA 1.5",
    "GSAI-ML__LLaDA-8B-Instruct": "LLaDA 8B Instruct",
    "Dream-org__Dream-v0-Instruct-7B": "Dream 7B",
    "apple__DiffuCoder-7B-Instruct": "DiffuCoder 7B",
    "Gen-Verse__TraDo-4B-Instruct": "TraDo 4B",
    "Gen-Verse__TraDo-8B-Instruct": "TraDo 8B",
}

# Task display names
TASK_DISPLAY_NAMES = {
    "parallelbench_all": "Average (All Tasks)",
    "parallelbench_waiting_line_copy": "Copy",
    "parallelbench_waiting_line_insert_index": "Insert (Index)",
    "parallelbench_waiting_line_insert_random": "Insert (Random)",
    "parallelbench_waiting_line_remove_index": "Remove (Index)",
    "parallelbench_waiting_line_remove_random": "Remove (Random)",
    "parallelbench_waiting_line_replace_index": "Replace (Index)",
    "parallelbench_waiting_line_replace_random": "Replace (Random)",
    "parallelbench_waiting_line_reverse": "Reverse",
    "parallelbench_waiting_line_shuffle": "Shuffle",
    "parallelbench_waiting_line_sort": "Sort",
    "parallelbench_text_writing_paraphrasing": "Paraphrasing",
    "parallelbench_text_writing_summarization": "Summarization",
    "parallelbench_text_writing_words_to_sentence_easy": "Words→Sentence (Easy)",
    "parallelbench_text_writing_words_to_sentence_medium": "Words→Sentence (Medium)",
    "parallelbench_text_writing_words_to_sentence_hard": "Words→Sentence (Hard)",
    "parallelbench_puzzles_sudoku_n4": "Sudoku (4×4)",
    "parallelbench_puzzles_latin_square_n4": "Latin Square (4×4)",
}


# matplotlib rcParams for publication quality
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.dpi": 150,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 9,
        "lines.linewidth": 1.8,
        "lines.markersize": 6,
        "grid.alpha": 0.3,
        "savefig.bbox": "tight",
        "savefig.dpi": 300,
    }
)

## Chart Generation

In [ ]:
def plot_tps_vs_accuracy(
    df: pd.DataFrame,
    model: str,
    task: str,
    ax: plt.Axes | None = None,
    show_legend: bool = True,
    guidelines: list[float] | None = None,
) -> plt.Axes:
    """Plot TPS vs Accuracy for a single model and task.

    Each unmasking method is drawn as a separate line with markers.
    X-axis uses log2 scale (TPS values are powers of 2 for topk methods).
    """
    if guidelines is None:
        guidelines = [80, 60]

    if ax is None:
        _, ax = plt.subplots()

    task_df = df[(df["model"] == model) & (df["task"] == task)]

    methods = sorted(task_df["method"].unique())
    for idx, method in enumerate(methods):
        method_df = task_df[task_df["method"] == method].sort_values("tps")
        if method_df.empty:
            continue

        style = get_method_style(method, idx)
        ax.plot(
            method_df["tps"],
            method_df["score"],
            color=style["color"],
            marker=style["marker"],
            label=style["label"],
            zorder=3,
        )

    # Horizontal guidelines
    for gl in guidelines:
        ax.axhline(
            y=gl, color="gray", linestyle="--", linewidth=0.8, alpha=0.5, zorder=1
        )

    # Axis configuration
    ax.set_xscale("log", base=2)
    ax.set_xticks([1, 2, 4, 8, 16, 32])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlim(0.8, 40)
    ax.set_ylim(-5, 105)
    ax.set_xlabel("# Tokens per Step (TPS)")
    ax.set_ylabel("Accuracy (%)")

    model_display = MODEL_DISPLAY_NAMES.get(model, model)
    task_display = TASK_DISPLAY_NAMES.get(task, task)
    ax.set_title(f"{model_display} — {task_display}")

    ax.grid(True, alpha=0.3)

    if show_legend:
        ax.legend(loc="lower left", framealpha=0.9)

    return ax

## Generate All Charts (Inline)

In [ ]:
# Task order: avg first, then waiting_line, text_writing, puzzles
TASK_ORDER = [
    "parallelbench_all",
    # Waiting Line
    "parallelbench_waiting_line_copy",
    "parallelbench_waiting_line_insert_index",
    "parallelbench_waiting_line_insert_random",
    "parallelbench_waiting_line_remove_index",
    "parallelbench_waiting_line_remove_random",
    "parallelbench_waiting_line_replace_index",
    "parallelbench_waiting_line_replace_random",
    "parallelbench_waiting_line_reverse",
    "parallelbench_waiting_line_shuffle",
    "parallelbench_waiting_line_sort",
    # Text Writing
    "parallelbench_text_writing_paraphrasing",
    "parallelbench_text_writing_summarization",
    "parallelbench_text_writing_words_to_sentence_easy",
    "parallelbench_text_writing_words_to_sentence_medium",
    "parallelbench_text_writing_words_to_sentence_hard",
    # Puzzles
    "parallelbench_puzzles_sudoku_n4",
    "parallelbench_puzzles_latin_square_n4",
]


def get_task_list(df: pd.DataFrame) -> list[str]:
    """Return tasks in display order, limited to tasks present in the data."""
    available = set(df["task"].unique())
    ordered = [t for t in TASK_ORDER if t in available]
    # Append any tasks not in TASK_ORDER (future-proofing)
    remaining = sorted(available - set(ordered))
    return ordered + remaining

In [ ]:
models = sorted(filtered_df["model"].unique())
tasks = get_task_list(filtered_df)

for model in models:
    model_display = MODEL_DISPLAY_NAMES.get(model, model)
    print(f"\n{'=' * 60}")
    print(f"  {model_display}")
    print(f"{'=' * 60}")

    # Average chart (larger)
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_tps_vs_accuracy(filtered_df, model, "parallelbench_all", ax=ax)
    plt.tight_layout()
    plt.show()

    # Individual task charts in a grid
    individual_tasks = [t for t in tasks if t != "parallelbench_all"]
    if not individual_tasks:
        continue

    n_cols = 3
    n_rows = (len(individual_tasks) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, task in enumerate(individual_tasks):
        plot_tps_vs_accuracy(
            filtered_df,
            model,
            task,
            ax=axes_flat[i],
            show_legend=(i == 0),  # Legend only on first subplot
        )

    # Hide unused axes
    for j in range(len(individual_tasks), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f"{model_display} — Per-Task Results", fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()

## PDF Export

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

for model in models:
    model_display = MODEL_DISPLAY_NAMES.get(model, model)
    pdf_path = output_dir / f"{model}.pdf"

    with PdfPages(pdf_path) as pdf:
        # Page 1: Average chart
        fig, ax = plt.subplots(figsize=(10, 6))
        plot_tps_vs_accuracy(filtered_df, model, "parallelbench_all", ax=ax)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Page 2+: Individual tasks in grid
        individual_tasks = [t for t in tasks if t != "parallelbench_all"]
        if individual_tasks:
            n_cols = 3
            n_rows = (len(individual_tasks) + n_cols - 1) // n_cols
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
            axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

            for i, task in enumerate(individual_tasks):
                plot_tps_vs_accuracy(
                    filtered_df,
                    model,
                    task,
                    ax=axes_flat[i],
                    show_legend=(i == 0),
                )

            for j in range(len(individual_tasks), len(axes_flat)):
                axes_flat[j].set_visible(False)

            fig.suptitle(f"{model_display} — Per-Task Results", fontsize=15, y=1.01)
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    print(f"Saved: {pdf_path}")

print(f"\nAll PDFs saved to {output_dir.resolve()}")